# Notebook 2 — IndiaFinBench: Inter-Annotator Agreement Analysis

Reproduces the kappa analysis from Section 4 of the paper.
Run from `IndiaFinBench/`.

In [ ]:
import json
import re
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

try:
    from rapidfuzz import fuzz
except ImportError:
    raise ImportError('rapidfuzz required: pip install rapidfuzz')

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

ROOT    = Path('..')
QA_JSON = ROOT / 'annotation/raw_qa/indiafinbench_qa_combined_150.json'
ANN_CSV = ROOT / 'annotation/annotated/ai_annotator_answers.csv'

print('Files exist:')
for p in [QA_JSON, ANN_CSV]:
    print(f'  {p.name}: {p.exists()}')

In [ ]:
# Cell 2 — Load QA JSON and AI annotator answers CSV
with open(QA_JSON, encoding='utf-8') as f:
    qa_data = json.load(f)
qa_map = {item['id']: item for item in qa_data}

ann_df  = pd.read_csv(ANN_CSV)
ann_map = dict(zip(ann_df['id'], ann_df['your_answer'].fillna('')))

print(f'QA items loaded      : {len(qa_map)}')
print(f'Annotator answers    : {len(ann_map)}')
print(f'Matched IDs          : {len(set(qa_map) & set(ann_map))}')
print(f'\nAnnotator CSV columns: {list(ann_df.columns)}')
print(f'\nSample (first 3 rows):')
print(ann_df.head(3).to_string(index=False))

In [ ]:
# Cell 3 — Compute per-task agreement rates using the same logic as compute_kappa.py

def normalise(text):
    if not text: return ''
    t = str(text).lower().strip()
    t = re.sub(r'[^\w\s%.]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def extract_yn(text):
    t = normalise(text)
    if t.startswith('yes'): return 'yes'
    if t.startswith('no'):  return 'no'
    return 'unclear'

def fuzzy_match(a, b, threshold=0.72):
    na, nb = normalise(a), normalise(b)
    if not na or not nb: return False
    if na == nb: return True
    nums_a = set(re.findall(r'\d+[\d,]*\.?\d*', na))
    nums_b = set(re.findall(r'\d+[\d,]*\.?\d*', nb))
    if nums_a and nums_b and nums_a == nums_b: return True
    return fuzz.token_set_ratio(na, nb) / 100.0 >= threshold

TASK_ORDER = ['regulatory_interpretation', 'numerical_reasoning',
              'contradiction_detection',   'temporal_reasoning']

task_agreements  = {t: [] for t in TASK_ORDER}
task_ref_labels  = {t: [] for t in TASK_ORDER}
task_ann_labels  = {t: [] for t in TASK_ORDER}
detail_rows      = []

for iid, item in qa_map.items():
    task     = item['task_type']
    ref_ans  = item['answer']
    ann_ans  = ann_map.get(iid, '')

    if task == 'contradiction_detection':
        ref_label = extract_yn(ref_ans)
        ann_label = extract_yn(ann_ans)
    else:
        agree     = fuzzy_match(ref_ans, ann_ans)
        ref_label = agree
        ann_label = True

    agrees = (ref_label == ann_label)
    task_agreements[task].append(int(agrees))
    task_ref_labels[task].append(ref_label)
    task_ann_labels[task].append(ann_label)
    detail_rows.append({
        'id': iid, 'task_type': task,
        'difficulty': item.get('difficulty',''),
        'ref_answer': ref_ans[:80], 'ann_answer': ann_ans[:80],
        'agree': int(agrees),
    })

detail_df = pd.DataFrame(detail_rows)

print('Per-task agreement rates:')
print(f'  {"Task":<32}  {"N":>4}  {"Agree%":>7}  Kappa/Notes')
print(f'  {"-"*32}  {"-"*4}  {"-"*7}  {"-"*20}')
for task in TASK_ORDER:
    ag  = task_agreements[task]
    n   = len(ag)
    pct = sum(ag)/n*100 if n else 0
    note = '(agreement rate only — extractive)'
    if task == 'contradiction_detection':
        note = '(Kappa computed in cell 5)'
    print(f'  {task:<32}  {n:>4}  {pct:>6.1f}%  {note}')

In [ ]:
# Cell 4 — Bar chart: agreement rate per task type
task_labels_short = ['REG', 'NUM', 'CON', 'TMP']
agree_pcts = [
    sum(task_agreements[t]) / len(task_agreements[t]) * 100
    for t in TASK_ORDER
]

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#4472C4' if t != 'contradiction_detection' else '#ED7D31'
          for t in TASK_ORDER]
bars = ax.bar(task_labels_short, agree_pcts, color=colors, edgecolor='white')
ax.axhline(90, color='red', linestyle='--', linewidth=1.2, label='90% threshold')

for bar, val in zip(bars, agree_pcts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_ylim(0, 105)
ax.set_ylabel('Agreement Rate (%)')
ax.set_title('IndiaFinBench: Inter-Annotator Agreement by Task', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

overall = sum(sum(task_agreements[t]) for t in TASK_ORDER) \
        / sum(len(task_agreements[t]) for t in TASK_ORDER) * 100
print(f'Overall agreement: {overall:.1f}%')

In [ ]:
# Cell 5 — Contradiction detection: Cohen's kappa + bootstrap CI (n=1000)
np.random.seed(42)

con_ref = [str(x) for x in task_ref_labels['contradiction_detection']]
con_ann = [str(x) for x in task_ann_labels['contradiction_detection']]

kappa = cohen_kappa_score(con_ref, con_ann)
print(f'Contradiction detection Cohen\'s kappa: {kappa:.3f}')

# Bootstrap CI for kappa (n=1000)
n_boot   = 1000
boot_ks  = []
pairs    = list(zip(con_ref, con_ann))
n_con    = len(pairs)

for _ in range(n_boot):
    sample = [pairs[i] for i in np.random.randint(0, n_con, n_con)]
    r_boot, a_boot = zip(*sample)
    try:
        boot_ks.append(cohen_kappa_score(r_boot, a_boot))
    except Exception:
        pass

ci_lo = np.percentile(boot_ks, 2.5)
ci_hi = np.percentile(boot_ks, 97.5)
print(f'Bootstrap 95% CI: [{ci_lo:.3f}, {ci_hi:.3f}]')
print(f'n_con = {n_con}, n_boot = {n_boot}')

# Distribution plot
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(boot_ks, bins=30, color='#4472C4', edgecolor='white', alpha=0.85)
ax.axvline(kappa, color='red', linestyle='-', linewidth=1.5, label=f'Observed kappa={kappa:.3f}')
ax.axvline(ci_lo, color='orange', linestyle='--', linewidth=1.2, label=f'95% CI [{ci_lo:.3f}, {ci_hi:.3f}]')
ax.axvline(ci_hi, color='orange', linestyle='--', linewidth=1.2)
ax.set_xlabel("Cohen's kappa")
ax.set_ylabel('Bootstrap frequency')
ax.set_title('Bootstrap Distribution of Kappa (Contradiction Detection)', fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Show 5 disagreement examples + final summary matching Table 3
disagree_df = detail_df[detail_df['agree'] == 0].copy()
print(f'Total disagreements: {len(disagree_df)}/{len(detail_df)}')
print('\n5 disagreement examples:')
display_cols = ['id','task_type','difficulty','ref_answer','ann_answer']
print(disagree_df[display_cols].head(5).to_string(index=False))

print('\n' + '='*70)
print('IndiaFinBench — Inter-Annotator Agreement Summary (Table 3)')
print('='*70)
task_short_map = {
    'regulatory_interpretation': 'REG',
    'numerical_reasoning':       'NUM',
    'contradiction_detection':   'CON',
    'temporal_reasoning':        'TMP',
}
for task in TASK_ORDER:
    ag  = task_agreements[task]
    n   = len(ag)
    pct = sum(ag)/n*100
    k_str = f'{kappa:.3f} (bootstrap CI [{ci_lo:.3f},{ci_hi:.3f}])' \
            if task == 'contradiction_detection' else 'N/A (agreement rate only)'
    print(f'  {task_short_map[task]}  n={n:>3}  agree={pct:>5.1f}%  kappa={k_str}')

overall = sum(sum(v) for v in task_agreements.values()) \
        / sum(len(v) for v in task_agreements.values()) * 100
print(f'  Overall  n=150  agree={overall:.1f}%')
print('='*70)